# Employee Management System (SQLite + pandas)

This is the **database course version** of the assignment — not an ML project. It's in the portfolio mostly because it was part of my coursework and I wanted the resume folder to include some plain database work, not just modeling notebooks.

**Scope:** A minimal employee CRUD on top of SQLite. Add, update, delete, list employees, plus a couple of simple aggregate queries (count by department, average salary). Everything runs inside the notebook against a local `.db` file.

If you're scanning my portfolio looking for ML notebooks, skip this one. If you're looking at the database-fundamentals box, this is what I had time to put together.


In [1]:
import sqlite3
import os
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/employees")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DATA_DIR / "employees.db"

# Start fresh each run so the demo is deterministic. In a real app you obviously wouldn't drop the DB.
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
print(f"DB: {DB_PATH}")


DB: ../../data/employees/employees.db


## Schema

In [2]:
cur.execute('''
CREATE TABLE employees (
    employee_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name    TEXT NOT NULL,
    last_name     TEXT NOT NULL,
    email         TEXT UNIQUE NOT NULL,
    department    TEXT NOT NULL,
    salary        REAL NOT NULL CHECK (salary >= 0),
    hire_date     TEXT NOT NULL,
    active        INTEGER NOT NULL DEFAULT 1
)
''')

cur.execute('''
CREATE INDEX idx_dept ON employees(department)
''')

conn.commit()
print("Created table employees with index on department.")


Created table employees with index on department.


## CRUD operations

In [3]:
def add_employee(first, last, email, dept, salary, hire_date):
    cur.execute(
        '''INSERT INTO employees (first_name, last_name, email, department, salary, hire_date)
           VALUES (?, ?, ?, ?, ?, ?)''',
        (first, last, email, dept, salary, hire_date),
    )
    conn.commit()
    return cur.lastrowid

def update_employee(emp_id, **kwargs):
    if not kwargs:
        return 0
    cols = ", ".join(f"{k} = ?" for k in kwargs)
    cur.execute(f"UPDATE employees SET {cols} WHERE employee_id = ?", (*kwargs.values(), emp_id))
    conn.commit()
    return cur.rowcount

def delete_employee(emp_id, hard=False):
    # Soft-delete by default. Hard delete only if explicitly asked.
    if hard:
        cur.execute("DELETE FROM employees WHERE employee_id = ?", (emp_id,))
    else:
        cur.execute("UPDATE employees SET active = 0 WHERE employee_id = ?", (emp_id,))
    conn.commit()
    return cur.rowcount

def list_employees(active_only=True):
    q = "SELECT * FROM employees"
    if active_only:
        q += " WHERE active = 1"
    return pd.read_sql_query(q, conn)

print("CRUD functions defined.")


CRUD functions defined.


## Populate the table

In [4]:
rows = [
    ("Aisha",  "Patel",      "aisha.p@acme.com",   "Engineering", 95000,  "2021-03-15"),
    ("Marcus", "Lee",        "marcus.l@acme.com",  "Engineering", 110000, "2019-11-04"),
    ("Priya",  "Sharma",     "priya.s@acme.com",   "Data",        88000,  "2022-06-21"),
    ("Tom",    "Andersen",   "tom.a@acme.com",     "Sales",       72000,  "2020-01-08"),
    ("Linh",   "Nguyen",     "linh.n@acme.com",    "Data",        92000,  "2023-02-10"),
    ("Diego",  "Hernandez",  "diego.h@acme.com",   "Sales",       68000,  "2022-09-19"),
    ("Maya",   "Goldstein",  "maya.g@acme.com",    "HR",          61000,  "2021-08-02"),
    ("Yusuf",  "Okafor",     "yusuf.o@acme.com",   "Engineering", 102000, "2020-05-12"),
]
for r in rows:
    add_employee(*r)
print(f"Inserted {len(rows)} employees.")
list_employees()


Inserted 8 employees.


,employee_id,first_name,last_name,email,department,salary,hire_date,active
0,1,Aisha,Patel,aisha.p@acme.com,Engineering,95000.0,2021-03-15,1
1,2,Marcus,Lee,marcus.l@acme.com,Engineering,110000.0,2019-11-04,1
2,3,Priya,Sharma,priya.s@acme.com,Data,88000.0,2022-06-21,1
3,4,Tom,Andersen,tom.a@acme.com,Sales,72000.0,2020-01-08,1
4,5,Linh,Nguyen,linh.n@acme.com,Data,92000.0,2023-02-10,1
5,6,Diego,Hernandez,diego.h@acme.com,Sales,68000.0,2022-09-19,1
6,7,Maya,Goldstein,maya.g@acme.com,HR,61000.0,2021-08-02,1
7,8,Yusuf,Okafor,yusuf.o@acme.com,Engineering,102000.0,2020-05-12,1


## Update + soft delete + list

In [5]:
# Marcus got promoted
update_employee(2, salary=125000, department="Engineering")

# Tom left the company — soft delete (set active=0)
delete_employee(4)

# Active employees only
active = list_employees(active_only=True)
print("Active employees:", len(active))
active


Active employees: 7


,employee_id,first_name,last_name,email,department,salary,hire_date,active
0,1,Aisha,Patel,aisha.p@acme.com,Engineering,95000.0,2021-03-15,1
1,2,Marcus,Lee,marcus.l@acme.com,Engineering,125000.0,2019-11-04,1
2,3,Priya,Sharma,priya.s@acme.com,Data,88000.0,2022-06-21,1
3,5,Linh,Nguyen,linh.n@acme.com,Data,92000.0,2023-02-10,1
4,6,Diego,Hernandez,diego.h@acme.com,Sales,68000.0,2022-09-19,1
5,7,Maya,Goldstein,maya.g@acme.com,HR,61000.0,2021-08-02,1
6,8,Yusuf,Okafor,yusuf.o@acme.com,Engineering,102000.0,2020-05-12,1


## Aggregate queries

In [6]:
# average salary by department
dept_summary = pd.read_sql_query(
    '''SELECT department,
              COUNT(*) AS headcount,
              ROUND(AVG(salary), 0) AS avg_salary,
              ROUND(MIN(salary), 0) AS min_salary,
              ROUND(MAX(salary), 0) AS max_salary
       FROM employees
       WHERE active = 1
       GROUP BY department
       ORDER BY avg_salary DESC''',
    conn,
)
print("Department summary (active employees only):")
print(dept_summary.to_string(index=False))

# how many hires per year?
hires_by_year = pd.read_sql_query(
    '''SELECT substr(hire_date, 1, 4) AS year, COUNT(*) AS n_hires
       FROM employees
       GROUP BY year
       ORDER BY year''',
    conn,
)
print("\nHires per year:")
print(hires_by_year.to_string(index=False))


Department summary (active employees only):
 department  headcount  avg_salary  min_salary  max_salary
Engineering          3    107333.0     95000.0    125000.0
       Data          2     90000.0     88000.0     92000.0
      Sales          1     68000.0     68000.0     68000.0
         HR          1     61000.0     61000.0     61000.0

Hires per year:
year  n_hires
2019        1
2020        2
2021        2
2022        2
2023        1


## Closing notes

In [7]:
# In a real system the connection lifecycle is managed by the web framework, not the notebook
conn.close()
print(f"Closed connection. DB lives at: {DB_PATH}")


Closed connection. DB lives at: ../../data/employees/employees.db


## What this notebook does *not* cover

The instructor's marking rubric for the original assignment also included transaction handling, role-based access, and a small CLI. I've kept those out of the notebook because the notebook form is awkward for interactive CLIs. The SQL itself is the part that maps cleanly to this format.

The reason this is in the portfolio is for completeness — to show that the database course version of an "employee system" exists alongside the ML notebooks. If a recruiter is looking for ORM/SQL fluency, this is the artifact. If they're looking for modeling, every other notebook in the folder is more relevant.
